In [ ]:
ALPHABET = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-_"

DIRS = [
    (-1,-1),(-1,0),(-1,1),
    (0,-1),        (0,1),
    (1,-1),(1,0),(1,1),
]

def init_board():
    board = {f"{c}{r}": "." for c in "abcdefgh" for r in "12345678"}
    board["d4"] = board["e5"] = "W"
    board["d5"] = board["e4"] = "B"
    return board

def opponent(p):
    return "W" if p == "B" else "B"

def sq_to_xy(sq):
    return ord(sq[0]) - 97, int(sq[1]) - 1

def xy_to_sq(x,y):
    return chr(97+x) + str(y+1)

def in_bounds(x,y):
    return 0 <= x < 8 and 0 <= y < 8

def flips(board, player, sq):
    if board[sq] != ".": return []
    x,y = sq_to_xy(sq)
    opp = opponent(player)
    res = []

    for dx,dy in DIRS:
        nx,ny = x+dx, y+dy
        tmp = []
        while in_bounds(nx,ny):
            s = xy_to_sq(nx,ny)
            if board[s] == opp:
                tmp.append(s)
            elif board[s] == player:
                if tmp: res += tmp
                break
            else:
                break
            nx += dx; ny += dy
    return res

def legal_moves(board, player):
    return [sq for sq in board if flips(board, player, sq)]

def sort_key(sq):
    # a1 → h8
    return (ord(sq[0]), int(sq[1]))

def apply_move(board, player, move):
    f = flips(board, player, move)
    if not f: raise ValueError("illegal")
    board = board.copy()
    board[move] = player
    for s in f:
        board[s] = player
    return board

def next_player(board, player):
    opp = opponent(player)
    if legal_moves(board, opp): return opp
    if legal_moves(board, player): return player
    return None

def encode_base64(n):
    if n == 0: return ALPHABET[0]
    out = ""
    while n:
        n, r = divmod(n, 64)
        out = ALPHABET[r] + out
    return out

def encode_game(game):
    moves = [game[i:i+2] for i in range(0,len(game),2)]
    board = init_board()
    player = "B"

    rank = 1
    mult = 1

    for m in moves:
        legal = sorted(legal_moves(board, player), key=sort_key)
        base = len(legal)
        idx = legal.index(m)

        rank += idx * mult
        mult *= base

        board = apply_move(board, player, m)
        player = next_player(board, player)
        if player is None: break
        print(f"Move: {m}, Legal: {legal}, Rank: {rank}, Mult: {mult}")
    return encode_base64(rank)

In [5]:
if __name__ == "__main__":
    game = "f5d6c3f3"
    encoded = encode_game(game)
    print("encoded:", encoded)

Move: f5, Legal: ['c4', 'd3', 'e6', 'f5'], Rank: 3, Mult: 4
Move: d6, Legal: ['d6', 'f4', 'f6'], Rank: 3, Mult: 12
Move: c3, Legal: ['c3', 'c4', 'c5', 'c6', 'c7'], Rank: 3, Mult: 60
Move: f3, Legal: ['d3', 'f3', 'f4', 'g5'], Rank: 63, Mult: 240
encoded: _
